In [1]:
import time
from qc_executor import Executor, QuantumCircuit, QuantumOperator, Parameters
from qc_executor.pauli_propagation import PauliPropagationExecutor, PermutationSymmetry

%matplotlib inline

In [2]:
x = Parameters("x", 1)
p = Parameters("p", 2)

qc = QuantumCircuit(2)
qc.h(0)
qc.ryy(0, 1, p[0] * x[0])

In [ ]:
p_obs = Parameters("p_obs", 2)
observable = QuantumOperator(["ZI", "IZ"], [p_obs[0], p_obs[1]])

In [ ]:
executor: PauliPropagationExecutor = Executor.create("pauli_propagation", seed=0, shots=10000)

pp_circuit = executor.transpile_circuit(qc)
pp_observable = executor.transpile_operator(observable)

In [ ]:
result = executor.expectation_value(
    pp_circuit,
    pp_observable,
    x=[0.1],
    p=[0.3],
    p_obs=[0.5, 0.6],
)
print("Expectation value:", result)

Expectation value: 0.5997300202493925


In [ ]:
v = executor.expectation_value_derivatives(
    pp_circuit,
    pp_observable,
    "x",
    "p",
    "p_obs",
    x=[0.1],
    p=[0.3],
    p_obs=[0.5, 0.6],
)
print(v)

{'x': array([-0.00539919]), 'p': array([-0.00179973]), 'pop': array([0.        , 0.99955003])}


In [7]:
qc_executor.statevector(
    pp_circuit,
    x=[0.8],
    p=[0.5],
)

array([0.69301172+0.j        , 0.        -0.14048043j,
       0.69301172+0.j        , 0.        +0.14048043j])

In [8]:
qc_executor.sample(
    pp_circuit,
    x=[0.8],
    p=[0.5],
)

{'10': 4815, '00': 4810, '11': 195, '01': 180}

## Symmetry Merging

In [ ]:
def symm_qc(num_qubits):
    qc = QuantumCircuit(num_qubits)
    for i in range(num_qubits):
        qc.h(i)
    for i in range(num_qubits):
        for j in range(i + 1, num_qubits):
            qc.ryy(i, j, 0.5)
    return qc


def symm_observable(num_qubits):
    paulis = []
    for i in range(num_qubits):
        paulis.append("I" * i + "Z" + "I" * (num_qubits - i - 1))
    return QuantumOperator(paulis, [1.0] * num_qubits)


header = "Num qubits | time without symmetry (s) | time with symmetry (s) | speedup | difference in results"
print(header)
print("".join("+" if c == "|" else "-" for c in header))
for num_qubits in range(5, 15):
    qc = executor.transpile_circuit(symm_qc(num_qubits))
    obs_no_sym = executor.transpile_operator(symm_observable(num_qubits))
    obs_sym = executor.transpile_operator(
        symm_observable(num_qubits), symmetry_strategy=PermutationSymmetry()
    )
    start = time.perf_counter()
    result_no_sym = executor.expectation_value(qc, obs_no_sym)
    time_no_sym = time.perf_counter() - start
    start = time.perf_counter()
    result_sym = executor.expectation_value(qc, obs_sym)
    time_sym = time.perf_counter() - start
    print(
        f"{num_qubits:<10} | {time_no_sym:>25.4f} | {time_sym:>22.4f} | {time_no_sym / time_sym:>7.2f} | {abs(result_no_sym - result_sym):>21.4e}"
    )

Num qubits | time without symmetry (s) | time with symmetry (s) | speedup | difference in results
-----------+---------------------------+------------------------+---------+----------------------
5          |                    0.0034 |                 0.0006 |    5.60 |            0.0000e+00
6          |                    0.0085 |                 0.0014 |    5.99 |            0.0000e+00
7          |                    0.0189 |                 0.0031 |    6.03 |            0.0000e+00
8          |                    0.0612 |                 0.0077 |    7.97 |            0.0000e+00
9          |                    0.1369 |                 0.0192 |    7.14 |            0.0000e+00
10         |                    0.3712 |                 0.0433 |    8.58 |            0.0000e+00
11         |                    0.8800 |                 0.1000 |    8.80 |            0.0000e+00
12         |                    2.2553 |                 0.2310 |    9.76 |            0.0000e+00
13         |        